# 09. Prediction Pipeline

This notebook builds a simple prediction pipeline for future groundwater level estimation.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

## Load Feature-Engineered Dataset

In [2]:
candidate_paths = [
    Path("../datasets/groundwater_feature_engineered.csv"),
    Path("datasets/groundwater_feature_engineered.csv"),
    Path("groundwater_feature_engineered.csv")
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("groundwater_feature_engineered.csv not found in expected paths.")

df = pd.read_csv(dataset_path, parse_dates=["Data Acquisition Time"])

print(f"Dataset path: {dataset_path.resolve()}")
print("Shape:", df.shape)

Dataset path: /content/groundwater_feature_engineered.csv
Shape: (69404, 25)


## Define Features and Chronological Split

The same feature list and 80-20 split are used for consistency.

In [3]:
target = "Groundwater Level Telemetry 6 Hourly (meter)"

features = [
    "Latitude", "Longitude", "RL_MSL",
    "Year", "Month", "Day", "Hour",
    "DayOfWeek", "WeekOfYear", "Quarter", "IsWeekend",
    "Lag_1", "Lag_4", "Lag_28",
    "RollingMean_4", "RollingStd_4",
    "Hour_sin", "Hour_cos", "Month_sin", "Month_cos",
    "Station_ID"
]

split_index = int(len(df) * 0.80)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

## Load Best Model if Available, Otherwise Train and Save

Model files are stored in the `models` folder for reuse.

In [4]:
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

best_model_path = model_dir / "best_model.pkl"
meta_path = model_dir / "best_model_meta.json"

def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred)
    }

if best_model_path.exists() and meta_path.exists():
    best_model = joblib.load(best_model_path)
    model_meta = json.loads(meta_path.read_text())
    best_model_name = model_meta.get("model_name", "Saved Model")
    print(f"Loaded saved best model: {best_model_name}")
else:
    candidates = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),
        "XGBoost": XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }

    rows = []
    fitted_models = {}

    for name, model in candidates.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        m = evaluate(y_test, pred)
        rows.append({"Model": name, **m})
        fitted_models[name] = model

    model_report = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    display(model_report.style.format({"MAE": "{:.4f}", "RMSE": "{:.4f}", "R²": "{:.4f}"}))

    best_model_name = model_report.loc[0, "Model"]
    best_model = fitted_models[best_model_name]

    joblib.dump(best_model, best_model_path)
    meta_path.write_text(json.dumps({"model_name": best_model_name, "features": features}, indent=2))

    print(f"Trained and saved best model: {best_model_name}")

# quick sanity check on current test split
test_pred = best_model.predict(X_test)
metrics = evaluate(y_test, test_pred)
print("\nCurrent model performance on test split:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

,Model,MAE,RMSE,R²
0,Linear Regression,2.5887,6.1434,0.8777
1,XGBoost,3.4198,7.4436,0.8204
2,Random Forest,4.0642,8.0264,0.7912


Trained and saved best model: Linear Regression

Current model performance on test split:
MAE: 2.5887
RMSE: 6.1434
R²: 0.8777


## Reusable Prediction Function

In [5]:
def predict_groundwater(input_df, model, feature_columns):
    """Predict groundwater level from a dataframe containing model features."""

    missing_cols = [col for col in feature_columns if col not in input_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    model_input = input_df[feature_columns].copy()
    preds = model.predict(model_input)

    output = input_df.copy()
    output["Predicted_Groundwater_Level_m"] = preds
    return output

## Example Predictions (Sample Rows from Dataset)

In [6]:
sample_input = X_test.head(10).copy()
prediction_output = predict_groundwater(sample_input, best_model, features)
prediction_output.head()

,Latitude,Longitude,RL_MSL,Year,Month,Day,Hour,DayOfWeek,WeekOfYear,Quarter,...,Lag_4,Lag_28,RollingMean_4,RollingStd_4,Hour_sin,Hour_cos,Month_sin,Month_cos,Station_ID,Predicted_Groundwater_Level_m
55523,12.86,77.79,880.0,2022,12,28,18,2,52,4,...,-24.918,-25.562,-24.90575,0.016049,-1.000000e+00,-1.836970e-16,-0.5,0.866025,20,-23.551430
55524,12.86,77.79,880.0,2022,12,29,0,3,52,4,...,-24.906,-25.538,-24.89925,0.014637,0.000000e+00,1.000000e+00,-0.5,0.866025,20,-23.532108
55525,12.86,77.79,880.0,2022,12,29,6,3,52,4,...,-24.916,-25.530,-24.89175,0.017443,1.000000e+00,6.123234e-17,-0.5,0.866025,20,-24.230234
55526,12.86,77.79,880.0,2022,12,29,12,3,52,4,...,-24.883,-25.499,-24.87925,0.010996,1.224647e-16,-1.000000e+00,-0.5,0.866025,20,-24.285827
55527,12.86,77.79,880.0,2022,12,29,18,3,52,4,...,-24.892,-25.498,-24.86550,0.027197,-1.000000e+00,-1.836970e-16,-0.5,0.866025,20,-23.474842


In [7]:
comparison_output = pd.DataFrame({
    "Actual": y_test.head(10).values,
    "Predicted": prediction_output["Predicted_Groundwater_Level_m"].values
})
comparison_output["Absolute_Error"] = np.abs(comparison_output["Actual"] - comparison_output["Predicted"])
comparison_output

,Actual,Predicted,Absolute_Error
0,-24.892,-23.551430,1.340570
1,-24.876,-23.532108,1.343892
2,-24.866,-24.230234,0.635766
3,-24.828,-24.285827,0.542173
4,1.000,-23.474842,24.474842
5,-24.834,-17.084860,7.749140
6,-24.834,-23.102122,1.731878
7,1.000,-23.164091,24.164091
8,-24.796,-12.273908,12.522092
9,-24.732,-22.060433,2.671567


## Single Input Example

This format can be used later when new feature rows are prepared from incoming telemetry data.

In [8]:
single_input = X_test.head(1).copy()
single_pred = predict_groundwater(single_input, best_model, features)
single_pred[["Predicted_Groundwater_Level_m"]]

,Predicted_Groundwater_Level_m
55523,-23.55143


## Conclusion

In this notebook, a reusable groundwater prediction pipeline was created using the best-performing machine learning model. The pipeline accepts new data containing the same engineered features used during training and returns predicted groundwater levels.

A sample prediction and a single-input example were tested successfully, confirming that the workflow can be reused without retraining the model `Predicted_Groundwater_Level_m`. This makes the project ready for future groundwater level prediction as new monitoring data becomes available.